# Plan1 training — BiLSTM+RAYEN imputers, 21-feature renewables dataset

Trains the three plan1 arms (`baseline`, `cost`, `size_aware`) on CUDA per `plan1.md`:
epochs 30 / patience 10, train+val loss = MSE, per-epoch history persisted to JSON.
Data preprocessing stays LOCAL — the 21-feature `net_dispatch_ren/prepared.npz` is
committed by `imputation/plan1_data.py`; this notebook only trains.

The final cell zips `imputation/results/plan1/` (weights + history + loss curves)
and downloads it — unzip into the same path on the local machine.

The `optimization` (QP trained-through) arm is NOT here — it gets its own
section/notebook after everything else, per plan1.md §7.

In [ ]:
REPO = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""  # repo is PRIVATE: paste a fine-grained READ-ONLY token (Contents: read)
import os
os.chdir("/content")            # always anchor here first (prevents nested clones on re-run)
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.exists("/content/energy_modelling"):
    !git clone -q $url
os.chdir("/content/energy_modelling")
!git pull -q
!nvidia-smi -L
import numpy as np
z = np.load("data/preprocessed/hist/5min/net_dispatch_ren/prepared.npz")
print("ren flats:", z["Xtr_flat"].shape, "features:", len(z["feat_cols"]))

In [ ]:
# arm 1/3 — baseline (rayen_traj trained through, equal balance split)
!python3 imputation/plan1_train.py --arm baseline

In [ ]:
# arm 2/3 — cost (baseline + 0.1 x dispatch-cost term, charging signed negative)
!python3 imputation/plan1_train.py --arm cost

In [ ]:
# arm 3/3 — size_aware (level-proportional balance split, in-graph and at eval)
!python3 imputation/plan1_train.py --arm size_aware

In [ ]:
# loss curves (plan1.md §3) — one figure, all arms, train + val MSE
import json, matplotlib.pyplot as plt
from pathlib import Path
OUT = Path("imputation/results/plan1")
ARMS = ["baseline", "cost", "size_aware"]
COLORS = {"baseline": "#2a78d6", "cost": "#eb6834", "size_aware": "#1baf7a"}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for arm in ARMS:
    p = OUT / f"{arm}_history.json"
    if not p.exists():
        print("missing", p); continue
    h = json.loads(p.read_text())
    ep = range(1, h["epochs_run"] + 1)
    ax[0].plot(ep, h["train_mse"], color=COLORS[arm], lw=2, label=arm)
    ax[1].plot(ep, h["val_mse"], color=COLORS[arm], lw=2, label=arm)
    bi = int(np.argmin(h["val_mse"]))
    ax[1].scatter([bi + 1], [h["val_mse"][bi]], s=50, color=COLORS[arm], zorder=5)
for a, t in zip(ax, ["train MSE (recon term)", "val MSE (through the arm's map; dot = best epoch)"]):
    a.set_title(t, loc="left", fontsize=10); a.set_xlabel("epoch"); a.grid(alpha=0.3)
    a.legend(frameon=False)
fig.suptitle("plan1 — training dynamics (3 arms, 21-feature renewables data)")
fig.tight_layout()
fig.savefig(OUT / "loss_curves.png", dpi=140)
plt.show()

In [ ]:
# zip results (weights + history + loss curves) and download to the local machine;
# unzip into imputation/results/plan1/ locally, then run plan1_bench.py / plan1_figures.py
import shutil
from google.colab import files
shutil.make_archive("/content/plan1_results", "zip", "imputation/results/plan1")
files.download("/content/plan1_results.zip")